In [25]:
# ------------------------------------------------------------
# Import Required Libraries
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ------------------------------------------------------------
# Load Loan Dataset
# ------------------------------------------------------------

# Read borrower data into a pandas DataFrame
df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

# Display first few records
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [26]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               10000 non-null  int64  
 1   credit_lines_outstanding  10000 non-null  int64  
 2   loan_amt_outstanding      10000 non-null  float64
 3   total_debt_outstanding    10000 non-null  float64
 4   income                    10000 non-null  float64
 5   years_employed            10000 non-null  int64  
 6   fico_score                10000 non-null  int64  
 7   default                   10000 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 625.1 KB
None


In [27]:
print(df.describe())

        customer_id  credit_lines_outstanding  loan_amt_outstanding  \
count  1.000000e+04              10000.000000          10000.000000   
mean   4.974577e+06                  1.461200           4159.677034   
std    2.293890e+06                  1.743846           1421.399078   
min    1.000324e+06                  0.000000             46.783973   
25%    2.977661e+06                  0.000000           3154.235371   
50%    4.989502e+06                  1.000000           4052.377228   
75%    6.967210e+06                  2.000000           5052.898103   
max    8.999789e+06                  5.000000          10750.677810   

       total_debt_outstanding         income  years_employed    fico_score  \
count            10000.000000   10000.000000    10000.000000  10000.000000   
mean              8718.916797   70039.901401        4.552800    637.557700   
std               6627.164762   20072.214143        1.566862     60.657906   
min                 31.652732    1000.000000    

In [28]:
print(df["default"].value_counts())

default
0    8149
1    1851
Name: count, dtype: int64


In [29]:
# ------------------------------------------------------------
# Define Features and Target Variable
# ------------------------------------------------------------

# Remove customer identifier and target variable
# from feature matrix

X = df.drop(
    columns=["customer_id", "default"]
)

# Target variable:
# 1 = Default
# 0 = No Default

y = df["default"]

In [30]:
X

,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score
0,0,5221.545193,3915.471226,78039.38546,5,605
1,5,1958.928726,8228.752520,26648.43525,2,572
2,0,3363.009259,2027.830850,65866.71246,4,602
3,0,4766.648001,2501.730397,74356.88347,5,612
4,1,1345.827718,1768.826187,23448.32631,6,631
...,...,...,...,...,...,...
9995,0,3033.647103,2553.733144,42691.62787,5,697
9996,1,4146.239304,5458.163525,79969.50521,8,615
9997,2,3088.223727,4813.090925,38192.67591,5,596
9998,0,3288.901666,1043.099660,50929.37206,2,647


In [31]:
y

0       0
1       1
2       0
3       0
4       0
       ..
9995    0
9996    0
9997    0
9998    0
9999    0
Name: default, Length: 10000, dtype: int64

In [32]:
# ------------------------------------------------------------
# Split Dataset
# ------------------------------------------------------------

# Use 80% of observations for training
# and 20% for testing model performance

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [33]:
# ------------------------------------------------------------
# Train Logistic Regression Model
# ------------------------------------------------------------

# Logistic Regression is commonly used
# in credit risk modelling because it
# directly estimates probabilities

model = LogisticRegression(
    max_iter=1000
)

# Fit model to training data

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [34]:
# ------------------------------------------------------------
# Evaluate Model Performance
# ------------------------------------------------------------

# Generate default predictions
y_pred = model.predict(X_test)

# Generate probability estimates
y_prob = model.predict_proba(X_test)[:, 1]

# Accuracy measures overall correctness
print("Accuracy:",
      accuracy_score(y_test, y_pred))

# ROC-AUC measures the model's ability
# to distinguish defaulters from non-defaulters
print("ROC-AUC:",
      roc_auc_score(y_test, y_prob))

# Detailed classification metrics
print(classification_report(
    y_test,
    y_pred
))

Accuracy: 0.997
ROC-AUC: 0.9999784447023711
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1630
           1       0.99      0.99      0.99       370

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [35]:
# ------------------------------------------------------------
# Model Coefficients
# ------------------------------------------------------------

# Examine the importance and direction
# of each feature in the logistic model

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coefficients.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

,Feature,Coefficient
0,credit_lines_outstanding,7.200514
4,years_employed,-2.848240
5,fico_score,-0.033446
2,total_debt_outstanding,0.001483
1,loan_amt_outstanding,0.000399
3,income,-0.000322


In [36]:
# ------------------------------------------------------------
# Expected Loss Assumptions
# ------------------------------------------------------------

# Recovery rate specified in task
RECOVERY_RATE = 0.10

# Loss Given Default (LGD)
LGD = 1 - RECOVERY_RATE

In [37]:
# ------------------------------------------------------------
# Expected Loss Function
# ------------------------------------------------------------

# Expected Loss (EL) is calculated as:
#
# EL = PD × LGD × EAD
#
# where:
# PD  = Probability of Default
# LGD = Loss Given Default
# EAD = Exposure at Default (loan amount)

In [38]:
def calculate_expected_loss(
    credit_lines_outstanding,
    loan_amt_outstanding,
    total_debt_outstanding,
    income,
    years_employed,
    fico_score
):
    """
    Calculates the Probability of Default (PD)
    and Expected Loss (EL) for a borrower.
    """

    # Create borrower record
    borrower = pd.DataFrame({
        "credit_lines_outstanding": [credit_lines_outstanding],
        "loan_amt_outstanding": [loan_amt_outstanding],
        "total_debt_outstanding": [total_debt_outstanding],
        "income": [income],
        "years_employed": [years_employed],
        "fico_score": [fico_score]
    })

    # Estimate Probability of Default (PD)
    pd_estimate = model.predict_proba(borrower)[0][1]

    # Expected Loss formula:
    # EL = PD × LGD × EAD
    expected_loss = (
        pd_estimate *
        LGD *
        loan_amt_outstanding
    )

    return {
        "probability_of_default": round(float(pd_estimate), 4),
        "expected_loss": round(float(expected_loss), 2)
    }

In [39]:
# ------------------------------------------------------------
# Example Borrower
# ------------------------------------------------------------

# Estimate default probability and
# expected loss for a sample borrower

result = calculate_expected_loss(
    credit_lines_outstanding=2,
    loan_amt_outstanding=15000,
    total_debt_outstanding=30000,
    income=60000,
    years_employed=5,
    fico_score=650
)

print(result)

{'probability_of_default': 1.0, 'expected_loss': 13500.0}


In [40]:
# ------------------------------------------------------------
# Low-Risk Borrower Example
# ------------------------------------------------------------

# High income, long employment history,
# and strong credit score

print(calculate_expected_loss(
    credit_lines_outstanding=1,
    loan_amt_outstanding=5000,
    total_debt_outstanding=8000,
    income=100000,
    years_employed=10,
    fico_score=800
))

{'probability_of_default': 0.0, 'expected_loss': 0.0}


In [43]:
# ------------------------------------------------------------
# High-Risk Borrower Example
# ------------------------------------------------------------

# Lower income, limited employment history,
# high debt burden, and lower credit score

print(calculate_expected_loss(
    credit_lines_outstanding=8,
    loan_amt_outstanding=30000,
    total_debt_outstanding=60000,
    income=25000,
    years_employed=1,
    fico_score=500
))

{'probability_of_default': 1.0, 'expected_loss': 27000.0}
